# Low-Resource Indic Text Classifier: Exploratory Data Analysis (EDA)
## Multilingual Indic NLP with AI4Bharat IndicBERT

This notebook explores the **AI4Bharat IndicSentiment** benchmark dataset across **Tamil (தமிழ்)**, **Hindi (हिन्दी)**, and **Malayalam (മലയാളം)**.

### Objectives:
1. Inspect total sample counts and language representation.
2. Verify class balance (Positive vs. Negative sentiment).
3. Analyze sentence character and word length distributions.
4. Audit Unicode script integrity for low-resource Indian languages.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "indic_sentiment_raw.csv"

print(f"Loading dataset from: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, encoding="utf-8")
df.info()

### 1. Dataset Preview

In [ ]:
df.head(10)

### 2. Language and Sentiment Label Distribution

In [ ]:
print("Language Distribution:")
print(df["language"].value_counts())

print("\nSentiment Distribution:")
print(df["label"].value_counts())

print("\nCross-Tabulation (Language x Sentiment):")
crosstab = pd.crosstab(df["language"], df["label"])
display(crosstab)

### 3. Visualizing Class and Language Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Language breakdown
sns.countplot(data=df, x="language", ax=axes[0], palette="Blues_r")
axes[0].set_title("Distribution by Language", fontsize=13, weight="bold")
axes[0].set_ylabel("Number of Samples")

# Sentiment breakdown by language
df_plot = df.copy()
df_plot["label_norm"] = df_plot["label"].str.lower()
sns.countplot(data=df_plot, x="language", hue="label_norm", ax=axes[1], palette={"positive": "#22C55E", "negative": "#EF4444"})
axes[1].set_title("Sentiment Balance per Language", fontsize=13, weight="bold")
axes[1].set_ylabel("Number of Samples")
axes[1].legend(title="Sentiment")

plt.tight_layout()
plt.show()

### 4. Text Length Analysis (Word Count & Character Count)

In [ ]:
df["word_count"] = df["text"].astype(str).apply(lambda t: len(t.split()))
df["char_count"] = df["text"].astype(str).apply(len)

print("Word Count Summary:")
display(df.groupby("language")["word_count"].describe())

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=df, x="word_count", hue="language", element="step", kde=True, common_norm=False, ax=ax)
ax.set_title("Word Count Distribution per Indic Language", fontsize=13, weight="bold")
ax.set_xlabel("Word Count")
ax.set_ylabel("Density")
plt.show()

### 5. Unicode Native Script Verification
Confirm that Indic scripts (Tamil: U+0B80–U+0BFF, Devanagari: U+0900–U+097F, Malayalam: U+0D00–U+0D7F) are preserved without character corruption.

In [ ]:
for lang in ["Tamil", "Hindi", "Malayalam"]:
    sample = df[df["language"] == lang]["text"].iloc[0]
    print(f"\nSample {lang} Review:")
    print(sample)
    sample_unicodes = [f"U+{ord(c):04X}" for c in sample[:5]]
    print(f"Initial character Unicode code points: {sample_unicodes}")